# Train Model 1 in Colab, from Neon (events) + the bucket (screenshots)

Same training as `01_train_local.ipynb`, but the data comes from the cloud:
- **Neon PostgreSQL** (schema `drc`) holds the synced DRC events: windows, browser, keyboard/mouse,
  and the screenshot list.
- **The bucket** `desktop-research-data-collection` holds the screenshot images.

| Step | What happens |
|---|---|
| 0 | Settings + sign-in |
| 1 | Get the project code (GitHub, or a zip you upload) and install MobileCLIP |
| 2 | Read **only your participant's rows** from Neon, and only the columns training needs |
| 3 | Rebuild them as a DRC-shaped SQLite file, so the tested replay code runs unchanged |
| 4 | Copy your screenshots from the bucket onto Colab's temporary disk (gone when the session ends) |
| 5–11 | Replay → MobileCLIP embeddings (GPU) → features → labels → split → prototypes → train → grade |
| 12 | Zip the trained model and download it |

**Read-only:** this notebook never writes to Neon or to the bucket. Other participants' data is
never fetched, because every query filters on `PROFILE_ID`.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU* (embedding about 11k screenshots on CPU would take hours).
2. Add your Neon connection string as a **Colab secret** named `DATABASE_URL` (key icon in the
   left bar → *Add new secret* → turn on *Notebook access*). Never paste it into a cell.
3. The code must be reachable. Either **push** `context-importance-system` (including
   `python-worker/offline/`) to GitHub, or zip the `python-worker` folder (without `.venv`), add
   `csharp-collector/src/CsharpCollector/collector.config.json` to the zip, and upload it in step 1.

> Outputs show screenshots and app names. Don't share or commit this notebook with outputs.

## 0. Settings

In [ ]:
PROFILE_ID = "imeth-k7mx9q2n_b4r"   # whose data to train on (rows of other participants are never read)
DB_SCHEMA = "drc"
BUCKET = "desktop-research-data-collection"
GCP_PROJECT = "project-dd3132f1-b1c9-49ea-bb8"

CODE_SOURCE = "github"              # "github" or "zip" (upload python-worker.zip in step 1)
REPO_URL = "https://github.com/imethsuka/J26-IT-308.git"
REPO_BRANCH = "main"
WORKER_SUBDIR = "context-importance-system/python-worker"

FROM_DATE = None                    # e.g. "2026-09-20"; None = everything for PROFILE_ID
TO_DATE = None                      # e.g. "2026-10-19"
CACHE_EMBEDDINGS_ON_DRIVE = True    # keep only the embeddings (small numbers, not images) on Drive
TEST_FRACTION = 0.3
PROTOTYPE_K = 8
THRESHOLD = 0.5
ALSO_USE_LOCAL_BUNDLE = False       # also train on the bundle from 01_train_local.ipynb (upload it in step 9b)

In [ ]:
from pathlib import Path

from google.colab import auth

auth.authenticate_user()

# Screenshots: copied from the bucket onto this Colab machine's own disk. Deleted automatically when
# the session ends, so the bucket stays the only lasting copy.
IMAGES = Path("/content/screenshots") / PROFILE_ID      # flat: <screenshot_id>.jpg
WORK = Path("/content/work") / PROFILE_ID               # rebuilt DB, model zip (temporary too)

# Embeddings: the slow part to recompute, but small and not convertible back into images.
if CACHE_EMBEDDINGS_ON_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/context-importance") / PROFILE_ID
else:
    CACHE = WORK
for p in (IMAGES, WORK, CACHE):
    p.mkdir(parents=True, exist_ok=True)
print("screenshots (temporary):", IMAGES)
print("embedding cache        :", CACHE)

## 1. Code and packages

In [ ]:
import shutil
import subprocess
import sys
import zipfile

if CODE_SOURCE == "github":
    repo = Path("/content/J26-IT-308")
    if not repo.exists():
        url = REPO_URL
        try:
            from google.colab import userdata

            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            print("no GITHUB_TOKEN secret: cloning anonymously (only works for a public repo)")
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_BRANCH, url, str(repo)], check=True)
    WORKER_DIR = repo / WORKER_SUBDIR
else:
    from google.colab import files

    uploaded = files.upload()   # choose python-worker.zip
    WORKER_DIR = Path("/content/python-worker")
    shutil.rmtree(WORKER_DIR, ignore_errors=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content")
    if not (WORKER_DIR / "offline").exists():  # zipped one level deeper
        WORKER_DIR = next(Path("/content").glob("**/python-worker"))

assert (WORKER_DIR / "offline" / "training.py").is_file(), f"offline/ not found under {WORKER_DIR}: push or re-zip the latest code"
# The replay applies the collector's exclusion list; it lives next to python-worker.
COLLECTOR_CONFIG = WORKER_DIR.parent / "csharp-collector" / "src" / "CsharpCollector" / "collector.config.json"
if not COLLECTOR_CONFIG.is_file():  # zip upload: look for the copy you added to the zip
    COLLECTOR_CONFIG = next(Path("/content").glob("**/collector.config.json"), COLLECTOR_CONFIG)
assert COLLECTOR_CONFIG.is_file(), "add csharp-collector/src/CsharpCollector/collector.config.json to the zip"
sys.path.insert(0, str(WORKER_DIR))
print("code:", WORKER_DIR, "| exclusion config found:", COLLECTOR_CONFIG.is_file())

In [ ]:
%pip install -q lightgbm river timm open_clip_torch "sqlalchemy>=2" psycopg2-binary pyarrow
%pip install -q --no-deps git+https://github.com/apple/ml-mobileclip.git

# If pip upgraded numpy while Colab already had the old one loaded, later cells fail with errors
# like "cannot import name '_slice' from numpy._core.umath". Restarting fixes it; on the next
# run nothing needs upgrading, so this only happens once per fresh runtime.
import importlib.metadata

import numpy

if importlib.metadata.version("numpy") != numpy.__version__:
    print(f"numpy was upgraded ({numpy.__version__} -> {importlib.metadata.version('numpy')}). "
          "Colab is restarting the session: wait for it to reconnect, then Run all again.")
    import os

    os.kill(os.getpid(), 9)
print("packages ready, numpy", numpy.__version__)

In [ ]:
import urllib.request

import torch

CHECKPOINT = Path("/content/mobileclip_s0.pt")  # public Apple weights, re-downloaded each session (~216 MB)
if not CHECKPOINT.exists():
    urllib.request.urlretrieve(
        "https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt", CHECKPOINT
    )
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MobileCLIP-S0 {CHECKPOINT.stat().st_size // 1_000_000} MB | device: {DEVICE}")
if DEVICE == "cpu":
    print("No GPU: Runtime -> Change runtime type -> T4 GPU, or step 6 will be very slow.")

## 2. Read your rows from Neon
Only the columns the replay uses are fetched. Page URLs, page titles, hashes, metadata and
display names are left in the database.

In [ ]:
import pandas as pd
from google.colab import userdata
from sqlalchemy import create_engine, text

import re
from urllib.parse import quote, unquote

from sqlalchemy.engine import make_url


def clean_database_url(raw: str) -> str:
    """Accept the string however Neon's dashboard gave it: plain URL, `psql '...'`, quoted, or with
    a trailing newline. Returns a SQLAlchemy URL for psycopg2. Special characters in the password
    are percent-encoded."""
    s = raw.strip()
    found = re.search(r"postgres(?:ql)?(?:\+\w+)?://[^\s'\"]+", s)
    if not found:
        raise ValueError("DATABASE_URL secret doesn't contain a postgresql://... URL. "
                         "In Neon: Connect -> choose 'Connection string' (not psql) and copy it.")
    rest = re.sub(r"^postgres(?:ql)?(?:\+\w+)?://", "", found.group(0))
    if "@" in rest:  # user:password@host/db?options; the host never contains '@', so split on the last one
        credentials, host_part = rest.rsplit("@", 1)
        user, _, password = credentials.partition(":")
        rest = f"{user}:{quote(unquote(password), safe='')}@{host_part}"
    return "postgresql+psycopg2://" + rest


url = clean_database_url(userdata.get("DATABASE_URL"))
safe = make_url(url)
print(f"connecting to {safe.host} / database {safe.database} as {safe.username} "
      f"(password hidden), options {dict(safe.query)}")
engine = create_engine(url, pool_pre_ping=True)
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("connected")

COLUMNS = {
    "activity_events": ["event_id", "session_id", "timestamp", "event_type", "application_name",
                        "process_name", "window_title", "duration_ms"],
    "browser_events": ["browser_event_id", "session_id", "timestamp", "domain"],
    "interaction_events": ["interaction_id", "session_id", "timestamp", "interaction_type", "count"],
    "screenshots": ["screenshot_id", "session_id", "timestamp", "capture_reason", "file_reference", "file_size"],
}


def fetch(table: str) -> pd.DataFrame:
    cols = ", ".join(f'"{c}"' for c in COLUMNS[table])
    where, params = ["profile_id = :pid"], {"pid": PROFILE_ID}
    if FROM_DATE:
        where.append('"timestamp" >= :from_date')
        params["from_date"] = FROM_DATE
    if TO_DATE:
        where.append('"timestamp" < (CAST(:to_date AS date) + 1)')
        params["to_date"] = TO_DATE
    sql = text(f'SELECT {cols} FROM "{DB_SCHEMA}"."{table}" WHERE {" AND ".join(where)} ORDER BY "timestamp"')
    with engine.connect() as conn:
        chunks = list(pd.read_sql(sql, conn, params=params, chunksize=200_000))
    df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=COLUMNS[table])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df


neon = {table: fetch(table) for table in COLUMNS}
engine.dispose()
for table, df in neon.items():
    span = f"{df['timestamp'].min():%Y-%m-%d} -> {df['timestamp'].max():%Y-%m-%d}" if len(df) else "empty"
    print(f"{table:20} {len(df):8} rows   {span}")

## 3. Rebuild a DRC-shaped database
The replay code (`offline/drc_replay.py`) reads DRC's SQLite layout, so the Neon rows are written into
a small SQLite file with that layout. Timestamps are stored as UTC ISO text, as DRC stores them.
The screenshot's file name (`<screenshot_id>.jpg`) links it to its image in the bucket.

In [ ]:
import sqlite3

DRC_SQLITE = WORK / "neon_as_drc.db"
DRC_SQLITE.unlink(missing_ok=True)

shots = neon["screenshots"].dropna(subset=["file_reference"]).copy()
shots["local_file_path"] = shots["file_reference"].str.rsplit("/", n=1).str[-1]

tables = {
    "activity_events": neon["activity_events"],
    "browser_events": neon["browser_events"],
    "interaction_events": neon["interaction_events"],
    "screenshots": shots.drop(columns=["file_reference"]),
}
with sqlite3.connect(DRC_SQLITE) as conn:
    for name, df in tables.items():
        out = df.copy()
        # One fixed width, so ORDER BY on the text matches time order.
        out["timestamp"] = out["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%S.%f+00:00")
        out.to_sql(name, conn, index=False)
print("wrote", DRC_SQLITE, f"({DRC_SQLITE.stat().st_size / 1e6:.1f} MB)")

## 4. Take your screenshots from the bucket
They're copied onto this Colab machine's temporary disk. That's fast (inside Google's network) and
deleted automatically when the session ends. Files already copied in this session are skipped.

In [ ]:
# Same method as "PCCILF Collab used to import bucket data.ipynb": list everything under your
# profile folder, download in parallel, skip files already downloaded with the same size.
# One change: files go into one flat folder (<screenshot_id>.jpg), because the replay finds each
# screenshot by its file name.
import os
from concurrent.futures import ThreadPoolExecutor

from google.cloud import storage
from tqdm.auto import tqdm

PREFIX = f"{PROFILE_ID}/"          # gs://desktop-research-data-collection/imeth-k7mx9q2n_b4r/...
client = storage.Client(project=GCP_PROJECT)
blobs = [b for b in client.list_blobs(BUCKET, prefix=PREFIX) if b.name.endswith(".jpg")]
print(f"Found {len(blobs)} files under gs://{BUCKET}/{PREFIX}")

skipped = 0


def download(blob):
    global skipped
    path = os.path.join(IMAGES, blob.name.rsplit("/", 1)[-1])
    # skip if already downloaded (same size)
    if os.path.exists(path) and os.path.getsize(path) == blob.size:
        skipped += 1
        return
    blob.download_to_filename(path)


with ThreadPoolExecutor(max_workers=16) as ex:
    list(tqdm(ex.map(download, blobs), total=len(blobs), desc="Downloading", unit="file"))

on_disk = {p.stem for p in IMAGES.glob("*.jpg")}
listed = set(shots["screenshot_id"])
print(f"Done. Skipped {skipped} already-downloaded files.")
print(f"Neon screenshots with an image: {len(listed & on_disk)} of {len(listed)}; "
      f"images with no Neon row (can't be labelled, ignored): {len(on_disk - listed)}")

## 5. Replay
Each screenshot becomes one capture, with what the live collector would have known at that moment.
This project's exclusions apply: password managers and login pages, the lock screen, and browser
pages with an unknown website (fail closed).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from eval.model_comparison import AlwaysSaveScorer
from models.rule_based import RuleBasedImportanceScorer
from models.static_threshold import StaticThresholdScorer
from offline.drc_replay import Exclusions, replay_drc
from offline.training import (
    add_labels, build_dataset, embed_captures, evaluate_models, learn_prototypes, load_bundle,
    new_version_id, save_version, temporal_split, threshold_curve, train_model1, with_prototypes,
)

captures, replay_stats = replay_drc(DRC_SQLITE, exclusions=Exclusions.from_collector_config(COLLECTOR_CONFIG),
                                    screenshots_dir=IMAGES)
print(replay_stats.summary())

## 6. The eyes: MobileCLIP embeddings (GPU)
With `CACHE_EMBEDDINGS_ON_DRIVE`, embeddings are saved to Drive as they're made. A later session
(or an interrupted one) only embeds screenshots it hasn't seen before.

In [ ]:
embeddings = embed_captures(captures, CACHE / "embeddings.npz", checkpoint=str(CHECKPOINT), device=DEVICE)
print(f"{sum(c.screenshot_id in embeddings for c in captures)} of {len(captures)} captures embedded")

## 7. Features and 8. labels
Same as the local notebook: features replayed in time order, then "did you come back within 48 h?" as the label.

In [ ]:
dataset = add_labels(build_dataset(captures, embeddings))
labelled = dataset[dataset["label"].notna()]
print(f"captures {len(dataset)} | labelled {len(labelled)} | important {labelled['label'].mean():.0%} "
      f"| still waiting (last 48 h) {dataset['label'].isna().sum()}")

## 9b. (Optional) Add the local bundle
With `ALSO_USE_LOCAL_BUNDLE = True`, upload `bundle-<participant>-<date>.parquet` from
`%LOCALAPPDATA%\context-importance\drc-training\` (made by `01_train_local.ipynb`). Its rows are
added to the cloud rows. Screenshots present in both are kept once.

In [ ]:
if ALSO_USE_LOCAL_BUNDLE:
    from google.colab import files

    uploaded = files.upload()
    local = pd.concat([load_bundle(name) for name in uploaded], ignore_index=True)
    local = local[local["participant"] == PROFILE_ID].drop(columns=["participant"])
    dataset = (pd.concat([dataset, local], ignore_index=True)
               .sort_values("ts").drop_duplicates("screenshot_id", keep="first").reset_index(drop=True))
    dataset["id"] = np.arange(len(dataset))
    print(f"combined: {len(dataset)} captures, {int(dataset['label'].notna().sum())} labelled, "
          f"{dataset['ts'].min():%Y-%m-%d} -> {dataset['ts'].max():%Y-%m-%d}")

## 9. Split by time, then 10. prototypes from the training days only

In [ ]:
train, test = temporal_split(dataset, test_fraction=TEST_FRACTION)
for name, part in (("train", train), ("test", test)):
    print(f"{name:5}: {len(part):5} captures  {part['ts'].min():%d %b} -> {part['ts'].max():%d %b}  important {part['label'].mean():.0%}")
assert train["label"].nunique() == 2, "training data needs both important and unimportant captures"

proto = learn_prototypes(train, k=PROTOTYPE_K)
prototypes = proto.centroids if proto is not None else None
print("prototypes:", proto.counts if proto is not None else "not enough important embeddings")
train_p, test_p = with_prototypes(train, prototypes), with_prototypes(test, prototypes)

## 11. Train Model 1 and grade it on the test days

In [ ]:
model, summary = train_model1(train_p)
print(f"trained on {summary['n_train']} captures, classes {summary['class_counts']}")
pd.Series(model.feature_importances("gain")).head(15)[::-1].plot.barh(figsize=(7, 5), title="What Model 1 relies on (gain)")
plt.tight_layout()
plt.show()


def without_vision(df):
    out = df.copy()
    out["features"] = [{**f, **{k: 0 for k in f if k.startswith("vis_")}} for f in df["features"]]
    return out


model_no_eyes, _ = train_model1(without_vision(train_p))
models = {
    "Rules (live today)": RuleBasedImportanceScorer(),
    "Static: dwell >= 30 s": StaticThresholdScorer(),
    "Model 1 (LightGBM)": model,
    "Model 1 without eyes": model_no_eyes,
    "DRC: keep every screenshot": AlwaysSaveScorer(),
}
results = pd.concat(
    [evaluate_models(without_vision(test_p) if n == "Model 1 without eyes" else test_p, {n: m}, threshold=THRESHOLD)
     for n, m in models.items()],
    ignore_index=True,
)
display(results[["name", "precision", "recall", "f1", "keep_rate", "storage_reduction"]].round(2))

curve = threshold_curve(test_p, model)
ax = curve.plot(x="threshold", y=["precision", "recall", "keep_rate", "storage_saved"], figsize=(7, 4), marker=".")
ax.axvline(THRESHOLD, color="grey", linestyle="--")
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

## 12. Download the trained model
The zip contains `model.txt` (the trained Model 1), `metadata.json`, the prototypes and
`training_notes.json`. On your PC, unzip it into `python-worker\models\versions\`. Then in
`01_train_local.ipynb` set `REGISTER_EXISTING_VERSION = "<version>"` and `DEPLOY_TO_LIVE_DB = True`
to activate it. It is activated only if it beats what the live worker serves now.

In [ ]:
from google.colab import files

version_id = new_version_id("lightgbm-neon")
versions_dir = WORK / "versions"
target = save_version(model, proto, version_id, versions_dir=versions_dir, notes={
    "source": f"neon {DB_SCHEMA} + gs://{BUCKET}", "profile_id": PROFILE_ID,
    "from_date": FROM_DATE, "to_date": TO_DATE, "local_bundle": ALSO_USE_LOCAL_BUNDLE,
    "replay": replay_stats.summary().splitlines(),
    "train": [str(train["ts"].min()), str(train["ts"].max()), len(train)],
    "test": [str(test["ts"].min()), str(test["ts"].max()), len(test)],
    "threshold": THRESHOLD,
    "results": results.drop(columns=["latency_mean_ms", "latency_p95_ms"]).to_dict("records"),
})
archive = shutil.make_archive(str(WORK / version_id), "zip", root_dir=versions_dir, base_dir=version_id)
print("saved", target)
files.download(archive)